# Liquidity Data for the Sector ETFs

**Data:** daily Open, High, Low, Close and Volume for the 11 sector ETFs and SPY from Yahoo Finance, Dec 1, 2023 to Sep 30, 2026.

**Measures**
- **Dollar volume** = Close x Volume, the value traded each day
- **Amihud illiquidity** = average of |daily return| / dollar volume, the price move per dollar traded

Yahoo does not provide historical bid-ask spreads, so spreads are not included here.

In [1]:
# !pip install yfinance   # uncomment if running in Colab

import os
import pandas as pd
import yfinance as yf

SECTORS = ["XLC", "XLY", "XLP", "XLE", "XLF", "XLV", "XLI", "XLB", "XLRE", "XLK", "XLU"]
TICKERS = SECTORS + ["SPY"]

BUDGET = 1_000_000
C_MAX = 0.20

DATA_FILE = "data/etf_liquidity_daily.xlsx"

## 1. Data collection

In [2]:
if not os.path.exists(DATA_FILE):
    frames = []
    for t in TICKERS:
        h = yf.Ticker(t).history(start="2023-12-01", end="2026-10-01", auto_adjust=False)
        h.index = h.index.tz_localize(None).normalize()
        h["Ticker"] = t
        frames.append(h[["Ticker", "Open", "High", "Low", "Close", "Volume"]])
    raw = pd.concat(frames).reset_index().rename(columns={"index": "Date"})
    raw.to_excel(DATA_FILE, index=False)

raw = pd.read_excel(DATA_FILE, parse_dates=["Date"])
print(raw.shape)
raw.head()

(8508, 7)


,Date,Ticker,Open,High,Low,Close,Volume
0,2023-12-01,XLC,69.650002,69.957001,69.209999,69.910004,7708500
1,2023-12-04,XLC,69.000000,69.300003,68.750000,69.209999,6609500
2,2023-12-05,XLC,69.040001,69.485001,68.510002,68.889999,5984500
3,2023-12-06,XLC,69.300003,69.370003,68.629997,68.709999,4552500
4,2023-12-07,XLC,69.389999,70.514999,69.370003,70.150002,6122600


## 2. Data quality checks

In [3]:
print("Missing values:", raw.isna().sum().sum())
print("Duplicate (Date, Ticker) rows:", raw.duplicated(["Date", "Ticker"]).sum())
print("Zero-volume rows:", (raw["Volume"] == 0).sum())
print("Rows where High < Low:", (raw["High"] < raw["Low"]).sum())
raw.groupby("Ticker")["Date"].agg(["count", "min", "max"])

Missing values: 0
Duplicate (Date, Ticker) rows: 0
Zero-volume rows: 0
Rows where High < Low: 0


,count,min,max
Ticker,,,
SPY,709,2023-12-01,2026-09-30
XLB,709,2023-12-01,2026-09-30
XLC,709,2023-12-01,2026-09-30
XLE,709,2023-12-01,2026-09-30
XLF,709,2023-12-01,2026-09-30
XLI,709,2023-12-01,2026-09-30
XLK,709,2023-12-01,2026-09-30
XLP,709,2023-12-01,2026-09-30
XLRE,709,2023-12-01,2026-09-30


Five ETFs had a 2-for-1 split on Dec 5, 2025. Yahoo adjusts both Close and Volume for it, so dollar volume should not jump on that date. The ratios below compare average dollar volume in the 20 trading days after the split with the 20 days before it.

In [4]:
close  = raw.pivot(index="Date", columns="Ticker", values="Close")[TICKERS]
volume = raw.pivot(index="Date", columns="Ticker", values="Volume")[TICKERS]
dollar_vol = close * volume

split_day = pd.Timestamp("2025-12-05")
before = dollar_vol.loc[:split_day].iloc[-21:-1].mean()
after  = dollar_vol.loc[split_day:].iloc[:20].mean()
(after / before).round(2)

Ticker
XLC     0.92
XLY     0.64
XLP     0.76
XLE     0.93
XLF     0.85
XLV     0.58
XLI     0.65
XLB     0.69
XLRE    1.17
XLK     0.47
XLU     0.81
SPY     0.94
dtype: float64

## 3. Liquidity measures (estimation window, Jan 2024 to Dec 2025)

In [5]:
est_dv = dollar_vol.loc["2024-01":"2025-12"]
ret = close.pct_change().loc["2024-01":"2025-12"]

amihud = (ret.abs() / est_dv).mean() * 1e9   # return per $1B traded
median_dv = est_dv.median()
position = BUDGET * C_MAX

liquidity = pd.DataFrame({
    "median daily $ volume ($M)": median_dv / 1e6,
    "lowest daily $ volume ($M)": est_dv.min() / 1e6,
    "Amihud (return per $1B)": amihud,
    "$200k position as % of median": position / median_dv * 100,
}).sort_values("median daily $ volume ($M)")
liquidity.round(3)

,median daily $ volume ($M),lowest daily $ volume ($M),Amihud (return per $1B),$200k position as % of median
Ticker,,,,
XLRE,234.523,90.023,0.030,0.085
XLB,432.041,126.867,0.017,0.046
XLC,460.996,161.330,0.016,0.043
XLY,769.786,196.891,0.013,0.026
XLU,846.807,246.628,0.009,0.024
XLP,931.184,312.406,0.006,0.021
XLI,1163.569,352.564,0.006,0.017
XLV,1233.798,437.367,0.005,0.016
XLK,1274.770,487.140,0.008,0.016


## 4. Does liquidity matter for our portfolio?

The largest position allowed is 20% of the budget. The table shows the budget at which that position would equal 1% of an ETF's median daily dollar volume, and of its lowest daily dollar volume.

In [6]:
capacity = pd.DataFrame({
    "budget at 1% of median day ($M)": 0.01 * median_dv / C_MAX / 1e6,
    "budget at 1% of lowest day ($M)": 0.01 * est_dv.min() / C_MAX / 1e6,
}).loc[SECTORS].sort_values("budget at 1% of median day ($M)")
capacity.round(0)

,budget at 1% of median day ($M),budget at 1% of lowest day ($M)
Ticker,,
XLRE,12.0,5.0
XLB,22.0,6.0
XLC,23.0,8.0
XLY,38.0,10.0
XLU,42.0,12.0
XLP,47.0,16.0
XLI,58.0,18.0
XLV,62.0,22.0
XLK,64.0,24.0


## 5. Liquidity inputs for the model

$L_i$ is the average daily dollar volume over the estimation window, and $q_i = L_{min} / L_i$ is the illiquidity score used in the objective.

In [7]:
L = est_dv[SECTORS].mean()
q = L.min() / L
print("Trading days D:", len(est_dv))
pd.DataFrame({"L ($M)": L / 1e6, "q": q}).sort_values("q").round(3)

Trading days D: 502


,L ($M),q
Ticker,,
XLF,1963.411,0.132
XLK,1455.652,0.178
XLE,1384.051,0.187
XLV,1368.774,0.189
XLI,1288.687,0.201
XLP,988.779,0.261
XLU,886.606,0.292
XLY,886.384,0.292
XLC,487.369,0.530
